In [2]:
import pandas as pd
import glob

# Load only lightweight columns to avoid memory crash
COLS = [
    "conversation_hash", "model", "timestamp", "turn",
    "language", "toxic", "redacted", "state", "country", "hashed_ip"
]

files = sorted(glob.glob("../Data/WildChatData/*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True)

In [3]:
print("Shape:", df.shape)
print("\nColumn types:")
display(df.dtypes.to_frame("dtype"))
print("\nFirst 5 rows:")
display(df.head())
print("\nDataset info:")
df.info()
print("\nNumeric summary:")
display(df.describe())

Shape: (4214886, 10)

Column types:


,dtype
conversation_hash,str
model,str
timestamp,"datetime64[us, UTC]"
turn,int64
language,str
toxic,bool
redacted,bool
state,str
country,str
hashed_ip,str



First 5 rows:


,conversation_hash,model,timestamp,turn,language,toxic,redacted,state,country,hashed_ip
0,c9ec5b440fbdd2a269333dd241f32f64,gpt-4-0314,2023-04-09 00:02:53+00:00,1,English,False,False,Texas,United States,22fd87ba9b98f3d379b23c7b52961f2d4a8505127e58b3...
1,34f1581760df304d539e2fe4653b40d3,gpt-4-0314,2023-04-09 00:03:20+00:00,2,Spanish,False,False,A Coruña,Spain,58369722cd0bdf7fc027a67491ba65b74576df6994c36c...
2,cf1267ca6b2f6fccc9c36652a00059a1,gpt-4-0314,2023-04-09 00:04:52+00:00,1,English,False,False,Mecca Region,Saudi Arabia,8133108d1c433c180c6be8302dc5a6681f2bec980190a1...
3,7f1c97a4f873cda8106b010d040be078,gpt-4-0314,2023-04-09 00:06:29+00:00,1,Catalan,False,False,Barcelona,Spain,846e43fb5fbb4b8cfbafa17083387aad62e58f5fb23482...
4,e98d3e74c57f9a65261df393d9124ac2,gpt-4-0314,2023-04-09 00:06:49+00:00,1,English,False,False,Texas,United States,22fd87ba9b98f3d379b23c7b52961f2d4a8505127e58b3...



Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 4214886 entries, 0 to 4214885
Data columns (total 10 columns):
 #   Column             Dtype              
---  ------             -----              
 0   conversation_hash  str                
 1   model              str                
 2   timestamp          datetime64[us, UTC]
 3   turn               int64              
 4   language           str                
 5   toxic              bool               
 6   redacted           bool               
 7   state              str                
 8   country            str                
 9   hashed_ip          str                
dtypes: bool(2), datetime64[us, UTC](1), int64(1), str(6)
memory usage: 817.7 MB

Numeric summary:


,turn
count,4.214886e+06
mean,2.336699e+00
std,3.013436e+00
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,2.000000e+00
max,2.490000e+02


In [4]:
df.columns.tolist()

['conversation_hash',
 'model',
 'timestamp',
 'turn',
 'language',
 'toxic',
 'redacted',
 'state',
 'country',
 'hashed_ip']

In [5]:
# Frequency tables for categorical columns
print("=== Model usage ===")
print(df["model"].value_counts(), "\n")

print("=== Top languages ===")
print(df["language"].value_counts().head(20), "\n")

print("=== Top countries ===")
print(df["country"].value_counts().head(20), "\n")

print("=== Toxic flag ===")
print(df["toxic"].value_counts(), "\n")

print("=== Redacted flag ===")
print(df["redacted"].value_counts(), "\n")

print("=== Missing values per column ===")
print(df.isnull().sum())

=== Model usage ===
model
gpt-3.5-turbo-0613    1865533
gpt-3.5-turbo-0301     979674
gpt-4-1106-preview     499637
gpt-4-0125-preview     313266
gpt-3.5-turbo-0125     288934
gpt-4-0314             267832
gpt-4-0613                 10
Name: count, dtype: int64 

=== Top languages ===
language
English       2415321
Chinese        597201
Russian        436156
French         135045
Spanish        102369
German          84413
Arabic          59676
Portuguese      52001
Turkish         30561
Italian         25620
Vietnamese      23763
Persian         19576
Nolang          18575
Polish          18350
Japanese        17485
Maori           16425
Latin           16171
Korean          16075
Indonesian      15336
Sotho           10965
Name: count, dtype: int64 

=== Top countries ===
country
United States      862896
Russia             571152
China              507496
Hong Kong          235781
United Kingdom     155161
Germany            145340
France             130949
Japan               88263

In [6]:
# Save combined data as a single parquet file (faster + smaller than CSV)
df.to_parquet("../Data/WildChatData/wildchat_combined.parquet", index=False)

In [7]:
# Load from combined file (use this instead of cell 1 once the file exists)
df = pd.read_parquet("../Data/WildChatData/wildchat_combined.parquet")

In [8]:
# Filter to English conversations only, then take a random sample of 5000 rows
english_df = df[df["language"] == "English"]
sample = english_df.sample(n=5000, random_state=42)

# Save the sample as parquet and CSV
sample.to_parquet("../Data/WildChatData/wildchat_sample_5000.parquet", index=False)
sample.to_csv("../Data/WildChatData/wildchat_sample_5000.csv", index=False)

print(f"Full dataset:         {df.shape[0]:,} rows")
print(f"English only:         {english_df.shape[0]:,} rows")
print(f"Sample size:          {sample.shape[0]:,} rows")
print("Saved to wildchat_sample_5000.parquet and wildchat_sample_5000.csv")
display(sample.head())

Full dataset:         4,214,886 rows
English only:         2,415,321 rows
Sample size:          5,000 rows
Saved to wildchat_sample_5000.parquet and wildchat_sample_5000.csv


,conversation_hash,model,timestamp,turn,language,toxic,redacted,state,country,hashed_ip
2402694,44b3b1b80512703529a61bc8d02cb96d,gpt-4-1106-preview,2024-03-10 03:22:47+00:00,5,English,False,False,North Yorkshire,United Kingdom,92f5a7cf0a2e6fa4e56edbe8e17f4b6459ce89a5ec5343...
3106122,f7511ee6982287c06d25dc095f40a1f9,gpt-4-1106-preview,2023-12-04 14:37:43+00:00,1,English,False,False,Renfrewshire,United Kingdom,a7ff28aae005c589f1fcbf72d63a37aef13071d724e770...
795708,865902eb603e9f3c90a97d4ccf5727f2,gpt-4-0125-preview,2024-04-10 16:19:58+00:00,4,English,False,False,Leinster,Ireland,e6927b8de4c42731cedade3905d5cb675b58785f8b3cfe...
707939,dbb2cbdac0c7ad91037628aa744214e5,gpt-3.5-turbo-0125,2024-02-29 10:55:39+00:00,1,English,False,False,Hubei,China,15edc737622df84201c45a0531625f78d5d574b9cc2daa...
1541581,9811d4e7f5cbbecb346bb8471cd6986f,gpt-4-1106-preview,2024-02-27 07:00:55+00:00,1,English,False,False,NaN,United States,85b4898def511e276333408699cf3f4f59bac477ca9383...


In [9]:
# Re-attach conversations to the current English sample
sample_hashes = set(sample["conversation_hash"])

# Use the original numbered source files only — generated files don't have conversation column
source_files = sorted(glob.glob("../Data/WildChatData/[0-9][0-9][0-9][0-9].parquet"))

conv_frames = []
for f in source_files:
    chunk = pd.read_parquet(f)[["conversation_hash", "conversation"]]
    conv_frames.append(chunk[chunk["conversation_hash"].isin(sample_hashes)])

conv_df = pd.concat(conv_frames, ignore_index=True)
sample_with_conv = sample.merge(conv_df, on="conversation_hash", how="left")

# Save as parquet (preserves nested conversation structure)
sample_with_conv.to_parquet("../Data/WildChatData/wildchat_sample_5000_with_conv.parquet", index=False)

# Save as CSV — conversations will be stored as text representations of the list
sample_with_conv.to_csv("../Data/WildChatData/wildchat_sample_5000_with_conv.csv", index=False)

print(f"Rows saved: {sample_with_conv.shape[0]:,}")
print(f"Columns:    {sample_with_conv.columns.tolist()}")
print("Saved to wildchat_sample_5000_with_conv.parquet and wildchat_sample_5000_with_conv.csv")

KeyboardInterrupt: 

In [ ]:
non_english = df[df["language"] != "English"]

print(f"Total rows:       {len(df):,}")
print(f"English:          {len(english_df):,} ({len(english_df)/len(df)*100:.1f}%)")
print(f"Non-English:      {len(non_english):,} ({len(non_english)/len(df)*100:.1f}%)")
print()
print("Top non-English languages:")
display(non_english["language"].value_counts().head(15).to_frame("count"))

Total rows:       4,214,886
English:          2,415,321 (57.3%)
Non-English:      1,799,565 (42.7%)

Top non-English languages:


,count
language,
Chinese,597201
Russian,436156
French,135045
Spanish,102369
German,84413
Arabic,59676
Portuguese,52001
Turkish,30561
Italian,25620


: 

In [3]:
# -- Read combined data tagged parquet to verify tags are present
VERIFY_COLS = ["conversation_hash", "model", "language", "country", "tags"]
tagged_df = pd.read_parquet("../Data/WildChatData/combined_data_tagged.parquet", columns=VERIFY_COLS)
print("Tagged dataset shape:", tagged_df.shape)
print("Columns:", tagged_df.columns.tolist())
print("\nSample tagged rows:")
display(tagged_df.head())

Tagged dataset shape: (837989, 5)
Columns: ['conversation_hash', 'model', 'language', 'country', 'tags']

Sample tagged rows:


,conversation_hash,model,language,country,tags
0,c9ec5b440fbdd2a269333dd241f32f64,gpt-4-0314,English,United States,"[""sql database"", ""data science ml ai"", ""cybers..."
1,34f1581760df304d539e2fe4653b40d3,gpt-4-0314,Spanish,Spain,"[""marketing seo copywriting"", ""math algebra ca..."
2,cf1267ca6b2f6fccc9c36652a00059a1,gpt-4-0314,English,Saudi Arabia,"[""history civilizations"", ""health medical"", ""t..."
3,7f1c97a4f873cda8106b010d040be078,gpt-4-0314,Catalan,Spain,"[""world building lore""]"
4,e98d3e74c57f9a65261df393d9124ac2,gpt-4-0314,English,United States,"[""javascript frontend"", ""sql database"", ""cyber..."
